# Normalise Bulgarian text (no number expansion)

Cleans raw Bulgarian text exactly the way `split_by_sentences.py` does, **except** it does
**not** expand numbers to words (no `num2wordBg`). The result is, per output line, one
sentence reduced to lowercase Bulgarian letters and single spaces — dots, punctuation,
digits and every other symbol are dropped.

Pipeline (matching `text/bulgarian.py:normalize_text` + the sentence splitter):
1. Split each paragraph line into sentences at `. ! ? …`.
2. Lowercase, fold accented/foreign look-alikes onto canonical Bulgarian letters.
3. Keep only the 30 Bulgarian letters and single spaces (everything else → space).
4. Collapse whitespace runs to one space and 3+ identical letters to 2.
5. Skip sentences that normalise to nothing.

Output: one cleaned sentence per line.

## 1. Normalisation primitives

Copied verbatim from `text/bulgarian.py` so the notebook is self-contained (runs in Colab
too). The number-expansion step from `split_by_sentences.py` is intentionally omitted.

In [ ]:
import re

# The 30 letters of the modern Bulgarian alphabet (lowercase).
BG_ALPHABET = list("абвгдежзийклмнопрстуфхцчшщъьюя")

# Accented / foreign look-alikes folded onto a canonical Bulgarian letter.
_NORMALIZE = {
    "ѝ": "и",  # и with grave accent (U+045D)
}

_VALID = set(BG_ALPHABET)
_WHITESPACE_RE = re.compile(r"\s+")
# Bulgarian never has 3+ identical letters in a row; collapse such runs to 2.
_REPEAT_RE = re.compile(r"(.)\1{2,}")

# A sentence ends at . ! ? or … (possibly several), followed by whitespace.
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?…])\s+")


def normalize_text(text):
    """Lowercase, fold look-alikes, keep only Bulgarian letters and single spaces.
    Punctuation, digits and any other characters become spaces; whitespace runs
    collapse to one and runs of 3+ identical letters collapse to 2."""
    out = []
    for ch in text.lower():
        ch = _NORMALIZE.get(ch, ch)
        out.append(ch if ch in _VALID else " ")
    cleaned = _WHITESPACE_RE.sub(" ", "".join(out)).strip()
    return _REPEAT_RE.sub(r"\1\1", cleaned)


def normalise_to_sentences(raw_text):
    """Raw text -> list of cleaned, one-sentence-per-line strings.
    NOTE: numbers are NOT expanded; digits are simply dropped by normalize_text."""
    sentences = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        for chunk in _SENTENCE_SPLIT_RE.split(line):
            cleaned = normalize_text(chunk)
            if cleaned:
                sentences.append(cleaned)
    return sentences

## 2. Quick check on a pasted string

In [ ]:
sample = "«Шибил». Здравей, свят! Имам 3 ябълки и 5 круши? Еххххх."
for s in normalise_to_sentences(sample):
    print(s)

## 3. Normalise a text file

Set `INPUT` / `OUTPUT`. Defaults read `testText1.txt` next to this notebook and write
`testText1.normalized.nonumbers.txt` (one sentence per line).

In [ ]:
from pathlib import Path

INPUT = Path("testText1.txt")
OUTPUT = Path("testText1.normalized.nonumbers.txt")

raw = INPUT.read_text(encoding="utf-8")
sentences = normalise_to_sentences(raw)
OUTPUT.write_text("\n".join(sentences) + "\n", encoding="utf-8")

print(f"{len(sentences)} sentences -> {OUTPUT}")
for s in sentences[:10]:
    print(s)